# 一次观察构图、层级超图学习与位置检索

依据 `docs/algorithm/Alg_GraphandOptimizer.md`，当前支持固定尺度／方向的平移检索。
前半部分显式 `learn=False`，只检查原始观察的分区、采样和边；后半部分使用独立长期记忆，
先查询再学习，展示 GmemIII 多角色组成、独立观察证据和 GposIII 定位。

本次改动只做编译与静态检查，没有执行下面的单元或调参。输出已清空，避免把旧结果当作新实现结果。
稀疏查询会报告近似与预算截断；小图可手动开启密集平移对照。参数扫描默认关闭。

当前默认 CUDA。安装环境更新后须选择并重启 `consciousLinux (CUDA 12.8)` 内核，从顶部重新执行。
2026-09-15 的 CPU 中断运行已存档至 `results/cuda_environment_20260915/cpu_interrupted_notebook.ipynb`；当前输出清空以免误认为 GPU 测试结果。


In [ ]:
from pathlib import Path
import copy
import importlib
import json
import math
import sys
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from PIL import Image, ImageDraw
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/nns/memorygraphs/graph_memorypool_onceoptimizer.py').is_file())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import nns.memorygraphs.graph_memorypool_onceoptimizer as memory_module
importlib.reload(memory_module)
from nns.memorygraphs.graph_memorypool_onceoptimizer import (
    MemoryConfig, MultilevelCoordinator, prepare_feature_subspaces,
    check_written_geometry, reference_star_response,
)
from nns.cnns.features import RetinaModel, MultiScaleFeatureBank

# CUDA is required for this notebook run; never silently fall back to CPU.
DEVICE = torch.device('cuda:0')
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'build CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Select consciousLinux (CUDA 12.8), check GPU access, and restart the kernel.')
try:
    # is_available alone did not catch the former cu126 / sm_120 mismatch.
    cudnn_version = torch.backends.cudnn.version()  # fail before CNN if an old system library shadows the wheel
    with torch.no_grad():
        probe = torch.arange(16, device=DEVICE, dtype=torch.float32).reshape(4, 4)
        probe_result = probe @ probe.T
        torch.cuda.synchronize(DEVICE)
        assert probe_result.sum().item() == 3680.0
        conv_probe = torch.nn.functional.conv2d(
            torch.ones((1, 1, 8, 8), device=DEVICE), torch.ones((1, 1, 3, 3), device=DEVICE))
        torch.cuda.synchronize(DEVICE)
        assert conv_probe.mean().item() == 9.0
    properties = torch.cuda.get_device_properties(DEVICE)
    CUDA_ENV = {'python': sys.executable, 'torch': torch.__version__,
                'cuda_build': torch.version.cuda, 'cudnn_version': cudnn_version, 'device': str(DEVICE),
                'gpu': properties.name, 'capability': torch.cuda.get_device_capability(DEVICE),
                'compiled_arches': torch.cuda.get_arch_list(),
                'total_memory_mib': properties.total_memory / 1024 ** 2}
    display(CUDA_ENV)
    del probe, probe_result, conv_probe
except Exception as exc:
    raise RuntimeError('CUDA execution failed. RTX 5060 Ti requires an sm_120-compatible build '
                       '(this project uses PyTorch 2.10.0+cu128). Select consciousLinux (CUDA 12.8) to avoid system cuDNN conflicts, then restart the kernel.') from exc
# This controls residual CPU tensor work, not GPU parallelism or Python graph loops.
torch.set_num_threads(2)
OUTPUT = ROOT / 'results/onceoptimizer'
print('Project:', ROOT)
print('PyTorch:', torch.__version__, 'device:', DEVICE)

## 1. 输入与配置

默认使用仓库内图片，可以将 `IMAGE_PATH=None` 切换为合成图。
输入缩放后再送入现有 Retina；不改变 CNN 参数。首轮只启用梯度与 RGB，
curv、aps（aspect，亦接受 asp 别名）、ori 默认与边缘及颜色一起学习；`ENABLE_ADVANCED=False` 仅用于双模态对照。所有距离参数均使用特征图像素单位。

`PARAMS` 只覆盖本次配置。降低 `tau_grad_str` 等已有门控会同时影响可写位置与检索，
应单独记录；以下没有预先宣称任何配置最优。

### 图片说明（从左到右）

| 图片 | 含义与颜色 |
| --- | --- |
| CNN view | 预处理后的 RGB 图，后续所有点与边均使用这张图的像素坐标。 |
| Valid input mask | 白色为有效输入，黑色为填充或无效位置；并非物体分割结果。 |
| Gradient strength | 梯度强度，颜色越亮表示变化越强；色条给出实际数值，不是概率。 |

所有图的坐标均为 `(x, y)`：向右为 x 增大，向下为 y 增大。


In [ ]:
from nns.memorygraphs.graph_memorypool_onceoptimizer import configure_orientation_contract
# IMAGE_PATH = ROOT / 'src/picture/OIP-C.jpg'  # None: synthetic image
IMAGE_PATH = '/home/p/code/ILSVRC/Data/DET/train/ILSVRC2014_train_0006/ILSVRC2014_train_00060030.JPEG'
VIEW_SIZE = 256
ENABLE_ADVANCED = True
ADVANCED_SCALES = [1, 2, 4]
PARAMS = {
    'once_node_reuse_mode': 'sample_instance',
    'once_samples_per_region': 64,
    'once_region_budget': 256,
    'once_write_local_contacts': False,
}


def make_synthetic(size):
    image = Image.new('RGB', (size, size), (235, 235, 235))
    draw = ImageDraw.Draw(image)
    def box(x0, y0, x1, y1):
        return tuple(int(v * size) for v in (x0, y0, x1, y1))
    draw.rectangle(box(.06, .08, .37, .40), fill=(60, 120, 200))
    draw.rectangle(box(.62, .10, .88, .34), fill=(60, 120, 200))
    draw.ellipse(box(.10, .54, .44, .89), fill=(200, 100, 60))
    draw.ellipse(box(.20, .64, .34, .79), fill=(235, 235, 235))
    draw.arc(box(.52, .46, .94, .90), start=5, end=285, fill=(30, 30, 30), width=3)
    return image


if IMAGE_PATH is None:
    pil_image = make_synthetic(VIEW_SIZE)
else:
    pil_image = Image.open(IMAGE_PATH).convert('RGB')
    pil_image.thumbnail((VIEW_SIZE, VIEW_SIZE))
image_tensor = torch.from_numpy(np.asarray(pil_image).copy()).permute(2, 0, 1)[None].float() / 255.
original_hw = tuple(image_tensor.shape[-2:])
retina = RetinaModel(cropped_size=VIEW_SIZE, output_size=VIEW_SIZE,
                    edge_apply_gaussian=True, edge_gauss_kernel_size=5, edge_gauss_sigma=1.).eval().to(DEVICE)
feature_bank = MultiScaleFeatureBank(ADVANCED_SCALES).eval().to(DEVICE) if ENABLE_ADVANCED else None


@torch.no_grad()
def encode_image(tensor, pixel_valid=None):
    tensor = tensor.to(DEVICE)
    _, grad, _, _, _, cropped = retina(tensor, center_x=-1, center_y=-1)
    # Process a mask through the exact same cropping transform, including odd dimensions.
    source_mask = torch.ones_like(tensor[:, :1]) if pixel_valid is None else pixel_valid.to(DEVICE)
    valid = retina.preprocess(source_mask, -1, -1)[0, 0] > 0.999
    named = {'grad': grad, 'rgb': cropped}
    if feature_bank is not None:
        curvature, aspect, orientation, confidence = feature_bank(
            cropped, precomputed_derivs={'grad': grad}, return_confidence=True)
        named.update(curvature=curvature, aspect=aspect, orientation=orientation,
                     orientation_confidence=confidence)
    return named, cropped, valid


features, image_view, valid_mask = encode_image(image_tensor)


def make_config(overrides=None):
    cfg = MemoryConfig()
    # Copy mutable class defaults before any experimental edits.
    for name in dir(cfg):
        if not name.startswith('_') and isinstance(getattr(cfg, name), (dict, list)):
            setattr(cfg, name, copy.deepcopy(getattr(cfg, name)))
    if feature_bank is not None:
        configure_orientation_contract(cfg, len(ADVANCED_SCALES))
    cfg.device = DEVICE
    cfg.H, cfg.W = image_view.shape[-2:]
    for key, value in {**PARAMS, **(overrides or {})}.items():
        if not hasattr(cfg, key):
            raise ValueError('Unknown parameter: ' + key)
        setattr(cfg, key, copy.deepcopy(value))
    return cfg


cfg = make_config()
inputs = prepare_feature_subspaces(features, cfg)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image_view[0].permute(1, 2, 0).cpu().clamp(0, 1)); axes[0].set_title('CNN view')
axes[1].imshow(valid_mask.cpu(), cmap='gray', vmin=0, vmax=1); axes[1].set_title('Valid input mask')
strength_plot = axes[2].imshow(inputs[0][0, 0].cpu(), cmap='magma'); axes[2].set_title('Gradient strength')
for ax in axes:
    ax.axis('off')
fig.colorbar(strength_plot, ax=axes[2], fraction=.046, pad=.04, label='Gradient magnitude')
fig.text(.5, .02, 'Mask: white = valid, black = padding/invalid. Coordinates: x right, y down.', ha='center', fontsize=9)
plt.tight_layout(rect=(0, .07, 1, 1))
print('Original H,W:', original_hw, 'feature shape:', tuple(image_view.shape))
print('Modalities:', {m: tuple(x.shape) for m, x in inputs.items()})
print('Binary coarse filter:', cfg.graph_binary_coarse, 'bin widths:', cfg.graph_coarse_bin_width)
print('Named orientation is normalized by pi/2; canonical integer modality 4 uses radians.')
display([{'id': m, 'name': cfg.feature_specs[m]['name'],
          'topology': cfg.feature_specs[m]['topology'],
          'channels': x.shape[1], 'finite_fraction': float(torch.isfinite(x).float().mean()),
          'nan_count': int(torch.isnan(x).sum()), 'inf_count': int(torch.isinf(x).sum()),
          'min': float(torch.nan_to_num(x).min()), 'max': float(torch.nan_to_num(x).max()),
          'support_threshold': cfg.once_support_threshold[m],
          'seed_threshold': cfg.once_seed_threshold[m],
          'similarity_threshold': cfg.once_similarity_min[m],
          'sample_budget': cfg.once_sample_budget[m],
          'projection_weight': cfg.modality_weights[m]}
         for m, x in inputs.items()])

raw_diagnostics = []
for name in ('curvature', 'aspect', 'orientation', 'orientation_confidence'):
    if name not in features:
        continue
    for channel, plane in enumerate(features[name][0]):
        finite = plane[torch.isfinite(plane)]
        raw_diagnostics.append({'feature': name, 'scale': ADVANCED_SCALES[channel],
            'nan': int(torch.isnan(plane).sum()), 'inf': int(torch.isinf(plane).sum()),
            'min_finite': float(finite.min()) if finite.numel() else None,
            'max_finite': float(finite.max()) if finite.numel() else None})
display(raw_diagnostics)
print('Orientation layout: S angle channels (radians), followed by S gate-only confidence channels.')


## 2. 一次观察

以下单元每次执行都创建独立记忆。返回的 `report` 包含支持图、标签、区域、采样和边的真值坐标。
节点激活状态保持默认；Gpos 查询通过显式节点列表进行，避免与神经元不应期混淆。

In [ ]:
coordinator = MultilevelCoordinator(cfg)
start = time.perf_counter()
report = coordinator.controller.run_step(
    inputs, coordinator.gmem_i, coordinator.gmem_ii, coordinator.gmem_iii, coordinator.gpos,
    valid_mask=valid_mask, learn=False,
)
coordinator.last_report = report
print('Observation:', coordinator.controller.current_step, 'elapsed:', time.perf_counter() - start)
display(report.summary())
print('GmemI / GmemII / GmemIII:', len(coordinator.gmem_i.nodes),
      len(coordinator.gmem_ii.semantic_nodes), len(coordinator.gmem_iii.entity_nodes))
print(len(report.regions))
print('Example rejections:')
display(report.rejection_examples[:12])
display([{'modality': m, 'name': cfg.feature_specs[m]['name'],
          'writable_pixels': int(s.writable.sum()), 'support_pixels': int(s.support.sum()),
          'seeds': len(s.seeds),
          'regions': sum(r.modality_id == m for r in report.regions),
          'written_nodes': sum(n.modality_id == m for n in coordinator.gmem_i.nodes.values())}
         for m, s in report.supports.items()])


## 3. 支持、种子与区域分割

每个模态单独输出一行四张图。模态编号：0=梯度/边缘，1=RGB，2=曲率，3=长宽比，4=方向。

| 图片（从左到右） | 含义 | 颜色与叠加标记 |
| --- | --- | --- |
| Reliability | 用于分割的支持可靠度，不是 Gpos 相似度，也不是识别概率；高可靠度不一定通过写入门控。 | 色条范围 0～1，紫色低、黄色高。 |
| Writable gate | 能通过现有描述子门控的位置。 | 白色=可写，黑色=不可写。 |
| Support / seeds / events | 用于局部连通分割的支持位置，可能经过强度筛选和边缘细化，因而比可写集合更小。 | 白色=支持，黑色=非支持；**红色圆点=初始种子**；**青色圆点=事件候选**。 |
| Regions / anchors | 保留的区域及其实际采样锚点。 | 伪彩色=区域标签；黑色=未保留区域；**黄色 ×=区域锚点**。 |

事件候选包括方向不可靠点或分支事件，不应一概解释为已确认的角点/交叉点。
初始种子只用于分区标记；区域锚点是在分割后选择的，二者可以不同。构图器补选的区域种子可在 `r.seeds` 查看，第三张图只画 `support.seeds` 中的初始种子。

区域色彩只帮助区分标签，不表示原始颜色、语义类别或相似度。`tab20` 颜色有限，**同色不保证属于同一区域**，不同模态的颜色也不对应。黑色可能来自门控、小区域过滤或预算，详见 `report.rejected`。不同模态的区域允许空间重叠。

ori 新格式附带结构各向异性置信度，置信度只参与门控、不参与角度相似度；轨迹沿梯度法线的垂直方向连接。上采样使用二倍角圆周插值。所有尺度均须有可靠方向才写入完整描述子。切勿复用旧格式记忆，请重启内核从头构建。


In [ ]:
for mid, support in report.supports.items():
    labels = report.labels[mid]
    fig, axes = plt.subplots(1, 4, figsize=(17, 4))
    reliability_plot = axes[0].imshow(support.quality, cmap='viridis', vmin=0, vmax=1)
    axes[0].set_title(f'{mid}: reliability')
    axes[1].imshow(support.writable, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Writable gate')
    axes[2].imshow(support.support, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title('Support / seeds / events')
    if support.seeds:
        axes[2].scatter(np.asarray(support.seeds) % cfg.W, np.asarray(support.seeds) // cfg.W, s=12, c='red', label='Initial seed')
    ey, ex = np.nonzero(support.events)
    axes[2].scatter(ex, ey, s=10, c='cyan', label='Event candidate')
    axes[3].imshow(np.ma.masked_less(labels, 0), cmap='tab20', interpolation='nearest')
    axes[3].set_facecolor('black'); axes[3].set_title('Regions / anchors')
    for r in report.regions:
        if r.modality_id == mid and r.anchor is not None:
            axes[3].scatter(r.anchor % cfg.W, r.anchor // cfg.W, c='yellow', marker='x', s=24, label='Region anchor')
    fig.colorbar(reliability_plot, ax=axes[0], fraction=.046, pad=.04, label='Reliability (not probability)')
    for ax in axes[2:]:
        handles, names = ax.get_legend_handles_labels()
        unique = dict(zip(names, handles))
        if unique:
            ax.legend(unique.values(), unique.keys(), loc='upper center', bbox_to_anchor=(.5, -.03), fontsize=8)
    for ax in axes:
        ax.axis('off')
    fig.text(.5, .02, 'Binary maps: white = included. Region colors are labels (colors may repeat); black = no retained region.', ha='center', fontsize=9)
    plt.tight_layout(rect=(0, .17, 1, 1))
    plt.show()

## 4. 单区域检查

修改 `REGION_INDEX` 查看其他区域。星型锚点一定是实际采样位置；
`coverage_error` 使用区域邻接图的路径距离，`curve_error` 是有序曲线的折线误差。
无法可靠排序的线域会标记 `curve_order_unresolved`，不会冒充完成。

### 图片与标记说明

| 图片/标记 | 含义 |
| --- | --- |
| 左图 RGB 底图 | 当前区域在输入图中的位置。 |
| 左图红橙色半透明覆盖 | 当前选中区域的支持范围；不是该区域的原始颜色。 |
| 右图二值掩码 | 白色=当前区域，黑色=区域外；孔洞也显示为黑色。 |
| 两图的青色圆点 | 最终选中的采样位置；有可写原型后用于注册/引用 GmemI 节点。 |
| 两图的红色 ★ | 当前区域的星型锚点；也是采样位置之一，可能遮住同位置的青色圆点。 |

有锚点标记并不保证区域已经写入：若预算不足，仍可能有候选锚点而 `semantic_id=None`；实际状态查看下方字段。
`coverage_error` 的单位为特征图像素路径距离；`curve_error` 仅对线域有意义，二维区域的默认值不代表做过曲线拟合。


In [ ]:
REGION_INDEX = 53


def show_region(index):
    if not report.regions:
        print('No retained regions; inspect gates, rejections and budgets first.')
        return
    r = report.regions[min(max(0, index), len(report.regions) - 1)]
    mask = report.labels[r.modality_id] == r.region_id
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_view[0].permute(1, 2, 0).cpu().clamp(0, 1))
    axes[0].imshow(np.ma.masked_where(~mask, mask), alpha=.35, cmap='autumn')
    axes[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[0].plot([], [], color='orangered', linewidth=8, alpha=.35, label='Selected region overlay')
    for ax in axes:
        if r.samples:
            samples = np.asarray(r.samples)
            ax.scatter(samples % cfg.W, samples // cfg.W, s=5, c='cyan', label='Sample position')
        if r.anchor is not None:
            ax.scatter(r.anchor % cfg.W, r.anchor // cfg.W, marker='*', s=120, c='red', label='Region anchor')
        ax.set_xlim(0, cfg.W); ax.set_ylim(cfg.H, 0)
        ax.set_xlabel('x (feature pixels)'); ax.set_ylabel('y (feature pixels)')
        handles, names = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, names, loc='upper center', bbox_to_anchor=(.5, -.13), fontsize=8)
    axes[0].set_title(f'Region {r.region_id}, modality {r.modality_id}')
    axes[1].set_title('Samples and anchor')
    fig.text(.5, .02, 'Left: selected region over RGB. Right: white = selected region, black = outside / holes.', ha='center', fontsize=9)
    plt.tight_layout(rect=(0, .17, 1, 1)); plt.show()
    display({'region': r.region_id, 'modality': r.modality_id, 'support_dim': r.support_dim,
             'pixels': len(r.pixels), 'samples': len(r.samples), 'semantic_id': r.semantic_id,
             'completed': r.completed, 'coverage_error': r.coverage_error, 'curve_error': r.curve_error,
             'budget_truncated': r.budget_truncated, 'view_truncated': r.view_truncated,
             'closed': r.closed, 'reasons': r.reasons})


show_region(REGION_INDEX)

## 5. 星型与跨区域接触

左图显示区域星型，右图显示跨区接触与可选曲线局部边。所有端点均属于 GmemI；
区域星型与接触关系归不同 GmemII 容器。为可读性，绘图可截断，实际存储数量另行打印。

### 图片与线条说明

- **左图青色线段**：区域星型边，将锚点与该区域的外围采样点连接。线段代表存储的相对位置关系，不是图像轮廓或眼跳轨迹。
- **右图红色线段**：跨区域接触边，以及启用 `once_write_local_contacts` 后的曲线局部边。每条关系由独立接触容器组织。
- 两图底图均为输入 RGB。线段不画箭头，不能从外观判断边方向；方向查看 `source_xy → target_xy`。共位连接投影为零长度，可能不可见。
- 标题中 `edges` 为该类边的总数，`showing` 为实际绘制数；超过 `MAX_DRAW_EDGES` 的部分仍在记忆中。


In [ ]:
MAX_DRAW_EDGES = 2000
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax in axes:
    ax.imshow(image_view[0].permute(1, 2, 0).cpu().clamp(0, 1))
    ax.set_xlim(0, cfg.W); ax.set_ylim(cfg.H, 0)
for ax, is_star, color in zip(axes, (True, False), ('cyan', 'red')):
    selected = [e for e in report.edge_observations if (e['kind'] == 'region_star') == is_star]
    segments = [(e['source_xy'], e['target_xy']) for e in selected[:MAX_DRAW_EDGES]]
    if segments:
        ax.add_collection(LineCollection(segments, colors=color, linewidths=.6, alpha=.6))
    ax.plot([], [], color=color, label='Anchor-to-peripheral relation' if is_star else 'Region / local contact relation')
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.04), fontsize=9)
    ax.set_title(f'{"Region stars" if is_star else "Contacts"}: {len(selected)} edges, showing {len(segments)}')
fig.text(.5, .02, 'Lines encode stored displacement, not contours or saccades. Arrows are omitted; coincident edges may be invisible.', ha='center', fontsize=9)
plt.tight_layout(rect=(0, .11, 1, 1)); plt.show()
print('Registered sample instances:', len(report.sample_nodes), 'referenced node ids:', len(report.node_ids))
print('Region containers:', len(report.region_semantic_ids), 'contact containers:', len(report.contact_semantic_ids))

## 6. 写入几何核验

该单元使用构图真值，仅检查写入是否正确，不属于位置检索实验。
实例模式下误差应接近浮点误差；共位边使用对数距离下限，显示其小量编码误差。
原型复用对照中，边合并可能改变几何，应查看最大误差和自环比例。

In [ ]:
geometry = check_written_geometry(report, coordinator.gmem_ii)
errors = np.asarray([row['position_error'] for row in geometry], dtype=float)
print('Max position error:', errors.max() if len(errors) else None)
print('Coincident edges:', sum(row['coincident'] for row in geometry))
print('Collapsed instances:', sum(row['collapsed_instances'] for row in geometry))
print('Merged edges:', sum(row['merged_count'] > 1 for row in geometry))
print('Worst rows:')
display(sorted(geometry, key=lambda row: row['position_error'], reverse=True)[:10])

## 7. 单区域 GposI／正式 GposII 与参考评分

原图、平移图和半图交换分别查询冻结的原始观察记忆。正式 GposII 返回共同位姿下的响应图、
多个通过验证的峰、可评估覆盖率、命中覆盖率及几何残差；Reference star 仍保留为独立对照。

四列依次为输入、GposI、参考评分和正式 GposII。青色 × 是预期锚点，红色 + 是预测锚点；
色条是连续匹配分数，不是校准概率。没有通过覆盖／几何门限的峰不报告为已识别。
半图交换只有在确实破坏目标结构时才是负例。


In [ ]:
QUERY_REGION_INDEX = 5
SHIFT_XY = (18, 12)
REFERENCE_TOLERANCE = 1


def shifted(tensor, dx, dy, fill=0.):
    out = torch.full_like(tensor, fill)
    h, w = tensor.shape[-2:]
    sx0, sx1 = max(0, -dx), min(w, w - dx)
    sy0, sy1 = max(0, -dy), min(h, h - dy)
    if sx1 > sx0 and sy1 > sy0:
        out[..., sy0 + dy:sy1 + dy, sx0 + dx:sx1 + dx] = tensor[..., sy0:sy1, sx0:sx1]
    return out


def peak_info(response):
    if response is None or not response.numel():
        return {'point': None, 'score': None}
    value = response[0, 0]
    idx = int(value.argmax().item())
    return {'point': (idx % value.shape[1], idx // value.shape[1]), 'score': float(value.max().item())}


query_results = []
written_regions = [r for r in report.regions if r.semantic_id is not None]
if not written_regions:
    print('No written regions to query.')
else:
    query_region = written_regions[min(max(0, QUERY_REGION_INDEX), len(written_regions) - 1)]
    sem = coordinator.gmem_ii.semantic_nodes[query_region.semantic_id]
    ids = sorted(sem.related_node_ids())
    source = image_view.detach().clone()
    dx, dy = SHIFT_XY
    source_valid = valid_mask[None, None].float()
    split = cfg.W // 2
    permutation = torch.cat((torch.arange(split, cfg.W), torch.arange(split)))
    query_cases = {
        'original': (source, source_valid, (0, 0)),
        'translated': (shifted(source, dx, dy), shifted(source_valid, dx, dy), (dx, dy)),
        'half_swap': (source[..., permutation], source_valid[..., permutation], None),
    }
    before_counts = (len(coordinator.gmem_i.nodes), len(coordinator.gmem_ii.semantic_nodes),
                     coordinator.controller.current_step)
    for name, (pixels, mask, translation) in query_cases.items():
        query_features, query_view, query_valid = encode_image(pixels, mask)
        maps = coordinator.query_l1(query_features, ids, valid_mask=query_valid)
        reference = reference_star_response(maps, sem, REFERENCE_TOLERANCE)
        existing = coordinator.gpos.l2_structure_synthesis(
            maps, sem, gmem_i=coordinator.gmem_i, valid_mask=query_valid)
        peak = peak_info(reference)
        expected = None if translation is None else (
            query_region.anchor % cfg.W + translation[0], query_region.anchor // cfg.W + translation[1])
        expected_visible = (expected is not None and 0 <= expected[0] < cfg.W and 0 <= expected[1] < cfg.H)
        error = math.dist(peak['point'], expected) if peak['point'] is not None and expected_visible else None
        query_results.append({'case': name, 'semantic_id': sem.node_id,
                              'reference_peak': peak, 'expected_anchor': expected,
                              'anchor_error': error, 'gposII': None if existing is None else existing['diagnostics']})
        fig, axes = plt.subplots(1, 4, figsize=(19, 4))
        axes[0].imshow(query_view[0].permute(1, 2, 0).cpu().clamp(0, 1)); axes[0].set_title(name)
        anchor_map = maps.get(sem.anchor_id)
        if anchor_map is not None:
            anchor_plot = axes[1].imshow(anchor_map[0, 0].cpu(), cmap='magma')
            fig.colorbar(anchor_plot, ax=axes[1], fraction=.046, pad=.04, label='Feature response')
        axes[1].set_title('GposI anchor response')
        if reference is not None:
            reference_plot = axes[2].imshow(reference[0, 0].cpu(), cmap='magma')
            fig.colorbar(reference_plot, ax=axes[2], fraction=.046, pad=.04, label='Reference score')
        axes[2].set_title('Reference star (baseline)')
        if existing is not None:
            formal_plot = axes[3].imshow(existing['response'][0, 0].cpu(), cmap='magma', vmin=0, vmax=1)
            fig.colorbar(formal_plot, ax=axes[3], fraction=.046, pad=.04, label='Structural score')
            for hypothesis in existing['matches']:
                axes[3].scatter(*hypothesis.point, marker='+', c='red', s=70)
        axes[3].set_title('GposII verified translation')
        if expected_visible:
            axes[2].scatter(*expected, marker='x', c='cyan', s=60, label='Expected anchor (ground truth)')
            axes[2].legend(loc='upper center', bbox_to_anchor=(.5, -.04), fontsize=8)
        fig.text(.5, .02, 'Brighter = larger response; read each colorbar. Cyan x is expected position, not a predicted peak.', ha='center', fontsize=9)
        plt.tight_layout(rect=(0, .16, 1, 1)); plt.show()
    after_counts = (len(coordinator.gmem_i.nodes), len(coordinator.gmem_ii.semantic_nodes),
                    coordinator.controller.current_step)
    print('Memory counts before / after read-only query:', before_counts, after_counts)
    display(query_results)

## 8. 长期超图学习与重复证据

本节使用独立 `learner` 对指定目录进行多图片顺序学习，复用前面的 CNN、配置与 `shifted` 函数。
**先重启内核并从顶部运行，确认首单元 GPU 运算检查通过，再执行本节。**
本节要求 `cuda:0`，不会静默退回 CPU，也不调参。默认固定随机种子，将 659 张图片按文件划分为
593 张训练图与 66 张留出图；`MAX_IMAGES` 可限制试运行规模，`HOLDOUT_FRACTION=0` 可学习全部图片。
每次运行创建全新记忆与独立日志目录，不支持从日志恢复记忆。建议重启内核后按顺序执行，避免旧变量占用内存。

检验内容：
- 三层节点、II 层区域关系、III 层从属／布局关系、稳定实体数量及新增／复用／归并／候选截断随图片数的变化。
- 第一张图同 episode 再学一次；检查证据条目不超过一次独立机会，且支持总量与 episode 明细一致。
  分数支持可能从较低值改善至 1，因此不把任何支持增量都判为去重失败。相同文件字节使用相同 episode 哈希。
- 冻结记忆后复查最早的 8 张训练图，比较初学与最终激活的实体编号；编号保持率仅是遗忘线索，合并／剪枝也会改变编号。
- 留出图只查询，不学习；报告区域／实体激活、覆盖率、几何残差、检索预算截断。
  平移测试比较冻结模型在原图与平移图上的同编号根位置，报告匹配数、像素残差与 2px 内比例。
  这属于平移一致性检查：边缘裁剪可能移除结构，多实例使用最近同编号峰，不能视为带真值的定位准确率。
- 查询前后检查节点／关系数量和 III 层实体版本、命中、支持、机会计数；这不是完整图状态的逐字节不变性证明。

内存报告以 **MiB** 为单位：持久图 Python 对象与唯一 Tensor 存储的估算占用、当前进程 RSS、
每 0.1 秒采样的本次峰值、相对开始基线的峰值增量、系统剩余可用内存；CUDA 模式还显示
allocated/reserved 及峰值、设备剩余显存。Linux 内核 `VmHWM` 是整个内核进程生命周期峰值，
与本次采样峰值分别展示；采样可能漏掉短峰。RSS 包含前面单元的 CNN、观察池及分配器缓存，
不能全部归为模型参数；图估算不含全部原生分配器开销、Gpos 临时响应图或优化器工作副本。
**学习暂存可能复制长期池，因此运行峰值通常比持久图体积更能反映所需资源。** 图遍历自身也有开销。

不缓存全数据集特征或各次 `LearningResult`；逐图指标写入 `results/onceoptimizer/dataset_*/metrics.jsonl`，
同时保存数据划分与配置 `manifest.json`、最终／部分运行汇总 `summary.json` 和学习／资源曲线。
异常或中断会停止后续学习并标明失败位置；只有完成的图片计入训练数量。
没有语义标注、负例阈值校准及多随机顺序重复实验时，激活比例不能称作物体识别准确率，
一次完整运行也不足以证明模型普遍可靠。文件划分未按物体类别／近重复内容分组。

性能诊断补充：每张图片显示进度，耗时阶段每 30 秒输出心跳；`learn_start` 在进入学习前立即写日志。
`stage_seconds` 区分已有记忆查询、观察构图、区域匹配、图复制、关系生成、实体对齐及提交，
阶段边界同步 CUDA，以便计时包含实际设备工作。`work_counts` 显示区域—模板比较数、密集回退次数、
候选中心数、槽位×中心数以及局部响应图计算数。编码、初学复查、同 episode 重复学习与其余开销单独记录。
额外保存 `learning_stages.png`。CPU／NumPy 图遍历仍在主机上运行，GPU 切换不改变搜索复杂度。

算法优化默认启用：先判定可评估范围，再分批使用分数／覆盖上界早退；稀疏请求只计算局部像素响应，保持原响应插值坐标。
`work_counts` 新增 `early_evaluable_rejected`、`upper_bound_rejected`、`progressive_sampled_slot_centers`、`local_response_pixels` 等计数；`dense_slot_centers` 仍是未剪枝的名义候选量。
预算默认关闭（`graph_search_budget_seconds=None`、`graph_search_budget_templates=None`）。未来启用时，返回 `UNRESOLVED` 的观察不提交长期图，本测试会记录原因并停止，不能当作已完成训练继续计数。此预算仅协作式约束区域搜索，不是整步硬实时期限；未实现自动后台重试队列。

二进制粗筛默认开启且只作保守排除；`work_counts` 中查看 `coarse_classes`、`coarse_template_checks`、`coarse_templates_rejected` 和 `coarse_templates_kept`。新增每模态节点统计用于确认五种模态实际写入。旧输出来自修改前配置，重启内核并从头执行后再比较。


In [ ]:
from collections import Counter
# Dataset protocol (not optimizer tuning). Run this cell explicitly to start learning.
import gc
import hashlib
import random
import threading
from datetime import datetime

DATASET_DIR = Path('/home/p/code/ILSVRC/Data/DET/train/ILSVRC2014_train_0006')
MAX_IMAGES = None                   # None: all 659; a small integer: smoke run
HOLDOUT_FRACTION = 0.10             # 0: train all images, no held-out evaluation
DATASET_SEED = 20260915
REPLAY_PROBES = 8                   # first training images, revisited after all learning
REPORT_EVERY = 1
GRAPH_SIZE_EVERY = 20               # traversal is outside learn/query timing
HEARTBEAT_SECONDS = 30
SAMPLE_SECONDS = 0.10               # RSS sampling may miss shorter transient peaks
DATASET_SHIFT = (18, 12)            # encoded-view pixels; fixed translation only

if torch.device(DEVICE).type != 'cuda' or torch.device(cfg.device) != DEVICE:
    raise RuntimeError('Restart kernel and run from the top: DEVICE and cfg must both use cuda:0.')
if image_view.device != DEVICE or any(value.device != DEVICE for value in inputs.values()):
    raise RuntimeError('Earlier cells still contain CPU tensors; rerun CNN/configuration on CUDA first.')

paths = sorted(p for p in DATASET_DIR.rglob('*')
               if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
if not paths:
    raise FileNotFoundError(f'No images in {DATASET_DIR}')
random.Random(DATASET_SEED).shuffle(paths)
if MAX_IMAGES is not None:
    if MAX_IMAGES < 1:
        raise ValueError('MAX_IMAGES must be positive or None')
    paths = paths[:MAX_IMAGES]
if not 0 <= HOLDOUT_FRACTION < 1:
    raise ValueError('HOLDOUT_FRACTION must be in [0, 1)')
n_holdout = min(len(paths) - 1, max(1, round(len(paths) * HOLDOUT_FRACTION))) if HOLDOUT_FRACTION else 0
holdout_paths, train_paths = paths[:n_holdout], paths[n_holdout:]
run_dir = OUTPUT / ('dataset_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / 'manifest.json').write_text(json.dumps({
    'dataset': str(DATASET_DIR), 'seed': DATASET_SEED,
    'train': [str(p) for p in train_paths], 'holdout': [str(p) for p in holdout_paths],
    'view_size': VIEW_SIZE, 'device': str(DEVICE), 'cuda_environment': CUDA_ENV, 'params': PARAMS,
    'advanced': ENABLE_ADVANCED, 'advanced_scales': ADVANCED_SCALES,
    'holdout_fraction': HOLDOUT_FRACTION, 'shift': DATASET_SHIFT,
    'config': {k: getattr(cfg, k) for k in dir(cfg)
               if not k.startswith('_') and not callable(getattr(cfg, k))},
}, ensure_ascii=False, indent=2, default=str))
print(f'Train={len(train_paths)}, held out={len(holdout_paths)}; logs: {run_dir}')

MIB = 1024 ** 2
use_cuda = torch.device(DEVICE).type == 'cuda'
def synchronize_device():
    if use_cuda:
        torch.cuda.synchronize(DEVICE)

def resources():
    # Linux /proc: RSS=current process, VmHWM=kernel lifetime high-water mark.
    def kb_fields(path):
        fields = {}
        for line in Path(path).read_text().splitlines():
            key, _, value = line.partition(':')
            parts = value.split()
            if parts and parts[0].isdigit():
                fields[key] = int(parts[0]) / 1024  # kB -> MiB
        return fields
    proc, host = kb_fields('/proc/self/status'), kb_fields('/proc/meminfo')
    row = {'rss_mib': proc.get('VmRSS', 0), 'process_lifetime_peak_mib': proc.get('VmHWM', 0),
           'host_available_mib': host.get('MemAvailable', 0), 'host_total_mib': host.get('MemTotal', 0)}
    if use_cuda:
        free, total = torch.cuda.mem_get_info(DEVICE)
        row.update(cuda_allocated_mib=torch.cuda.memory_allocated(DEVICE) / MIB,
                   cuda_reserved_mib=torch.cuda.memory_reserved(DEVICE) / MIB,
                   cuda_peak_allocated_mib=torch.cuda.max_memory_allocated(DEVICE) / MIB,
                   cuda_peak_reserved_mib=torch.cuda.max_memory_reserved(DEVICE) / MIB,
                   cuda_free_mib=free / MIB, cuda_total_mib=total / MIB)
    return row

class ResourceSampler:
    def __init__(self):
        self.stop = threading.Event()
        self.peak = 0.
        self.minimum_available = float('inf')
        self.error = None
        self.last_heartbeat = time.perf_counter()
    def sample(self):
        r = resources()
        self.peak = max(self.peak, r['rss_mib'])
        self.minimum_available = min(self.minimum_available, r['host_available_mib'])
    def loop(self):
        try:
            while not self.stop.wait(SAMPLE_SECONDS):
                self.sample()
                now = time.perf_counter()
                if now - self.last_heartbeat >= HEARTBEAT_SECONDS:
                    print(f"[heartbeat] {current_phase}: {Path(current_path).name if current_path else '-'}; "
                          f"elapsed={now - run_started:.0f}s, peak RSS={self.peak:.0f} MiB", flush=True)
                    self.last_heartbeat = now
        except Exception as exc:
            self.error = repr(exc)
    def __enter__(self):
        self.sample()
        self.thread = threading.Thread(target=self.loop, daemon=True)
        self.thread.start()
        return self
    def __exit__(self, *args):
        self.stop.set()
        self.thread.join()
        self.sample()

def graph_objects():
    # Inspect persistent pools only, not CNN, controller reports or cached query maps.
    seen, stack = set(), [learner.gmem_i, learner.gmem_ii, learner.gmem_iii]
    while stack:
        obj = stack.pop()
        if id(obj) in seen:
            continue
        seen.add(id(obj))
        yield obj
        if isinstance(obj, dict):
            stack.extend(obj.keys()); stack.extend(obj.values())
        elif isinstance(obj, (list, tuple, set, frozenset)):
            stack.extend(obj)
        elif isinstance(obj, np.ndarray):
            if obj.base is not None:
                stack.append(obj.base)
        elif not isinstance(obj, torch.Tensor) and hasattr(obj, '__dict__') and not isinstance(obj, type):
            stack.append(vars(obj))

def graph_footprint_and_evidence():
    # Python shallow sizes + unique Torch storages; approximate, not allocator RSS.
    total, tensor_cpu, tensor_cuda, invalid = 0, 0, 0, 0
    storages = set()
    for obj in graph_objects():
        total += sys.getsizeof(obj)
        if isinstance(obj, torch.Tensor):
            storage = obj.untyped_storage()
            key = (str(obj.device), storage.data_ptr(), storage.nbytes())
            if key not in storages:
                storages.add(key)
                if obj.is_cuda:
                    tensor_cuda += storage.nbytes()
                else:
                    tensor_cpu += storage.nbytes()
        if all(hasattr(obj, name) for name in ('episodes', 'support', 'opportunities', 'raw_hits')):
            entries = list(obj.episodes.values())
            invalid += int(any(not (0 <= hit <= opportunity <= 1) for opportunity, hit in entries)
                           or not math.isclose(obj.support, sum(hit for _, hit in entries), abs_tol=1e-5)
                           or not math.isclose(obj.opportunities, sum(op for op, _ in entries), abs_tol=1e-5))
    return {'graph_cpu_estimate_mib': (total + tensor_cpu) / MIB,
            'graph_cuda_storage_mib': tensor_cuda / MIB, 'invalid_evidence_records': invalid}

def memory_counts():
    entities = learner.gmem_iii.entity_nodes.values()
    return {'gmem_i_by_modality': dict(Counter(n.modality_id for n in learner.gmem_i.nodes.values())),
            'gmem_ii_by_modality': dict(Counter(learner.gmem_i.nodes[n.anchor_id].modality_id for n in learner.gmem_ii.semantic_nodes.values() if n.kind == 'region')),
            'gmem_i': len(learner.gmem_i.nodes), 'gmem_ii': len(learner.gmem_ii.semantic_nodes),
            'gmem_iii': len(learner.gmem_iii.entity_nodes),
            'ii_relations': len(learner.gmem_ii.region_relations),
            'iii_memberships': sum(len(e.component_edges) for e in entities),
            'iii_relations': sum(len(e.relation_edges) for e in learner.gmem_iii.entity_nodes.values()),
            'stable_entities': sum(e.status == 'stable' for e in learner.gmem_iii.entity_nodes.values()),
            'entity_raw_hits': sum(e.evidence.raw_hits for e in learner.gmem_iii.entity_nodes.values()),
            'entity_support': sum(e.evidence.support for e in learner.gmem_iii.entity_nodes.values())}

def release_observation():
    learner.last_report = learner.controller.last_report = None
    for name in ('matrix1', 'matrix2', 'matrix3', 'matrix4', 'current_interest_map'):
        setattr(learner.controller, name, None)
    learner.controller.debug_optimizer = {}
    gc.collect()

def load_dataset_image(path):
    with Image.open(path) as source:
        picture = source.convert('RGB')
        picture.thumbnail((VIEW_SIZE, VIEW_SIZE))
        pixels = torch.from_numpy(np.asarray(picture).copy()).permute(2, 0, 1)[None].float() / 255.
    return encode_image(pixels)

def query_compact(features, mask):
    synchronize_device(); start = time.perf_counter()
    result = learner.query_hierarchy(features, valid_mask=mask, exact=False)
    synchronize_device()
    # Do not retain response maps, assignments, or large static bitmaps.
    rows = [m.summary() for m in result.entities]
    diagnostics = {k: result.diagnostics.get(k) for k in (
        'search_complete', 'assignment_search_complete', 'event_budget_dropped',
        'candidate_budget_dropped', 'posting_visits', 'geometry_evaluations')}
    return rows, {'query_seconds': time.perf_counter() - start, 'region_matches': len(result.regions),
                  'entity_matches': len(rows), **diagnostics}

def append_log(row):
    with (run_dir / 'metrics.jsonl').open('a') as stream:
        stream.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')

# Re-running this cell starts a fresh experiment and releases its previous learner.
learner = None
gc.collect()
synchronize_device()
if use_cuda:
    torch.cuda.reset_peak_memory_stats(DEVICE)
baseline_resources = resources()
learner = MultilevelCoordinator(copy.deepcopy(cfg))
learning_rows, evaluation_rows, probe_baselines = [], [], {}
run_status = 'running'
current_path, current_phase = None, 'initialization'
run_started = time.perf_counter()

with ResourceSampler() as monitor:
    try:
        for step, path in enumerate(train_paths, 1):
            current_path, current_phase = str(path), 'train:encode'
            start = time.perf_counter()
            batch_features, batch_view, batch_mask = load_dataset_image(path)
            synchronize_device()
            encode_seconds = time.perf_counter() - start
            # Content hash: repeated bytes/epochs share evidence; crops share their source episode.
            episode = hashlib.sha256(path.read_bytes()).hexdigest()
            synchronize_device(); learn_start = time.perf_counter()
            current_phase = 'train:learn_view'
            append_log({'phase': 'learn_start', 'step': step, 'path': str(path),
                        'run_elapsed_seconds': learn_start - run_started})
            learned = learner.learn_view(batch_features, source_id=str(path),
                                         episode_id=episode, valid_mask=batch_mask)
            synchronize_device()
            diag = learned.diagnostics
            if diag.get('learning_status') == 'UNRESOLVED':
                append_log({'phase': 'unresolved', 'step': step, 'path': str(path), **diag})
                raise RuntimeError('Region search unfinished; no graph commit. Retry this image with more budget: '
                                   + diag.get('budget_reason', 'unknown'))
            def amount(value):
                return len(value) if isinstance(value, (list, tuple, dict, set)) else value
            row = {'phase': 'train', 'step': step, 'path': str(path),
                   'learn_seconds': time.perf_counter() - learn_start,
                   'encode_seconds': encode_seconds,
                   'stage_seconds': dict(diag.get('stage_seconds', {})),
                   'work_counts': dict(diag.get('work_counts', {})),
                   'prior_query_seconds': learned.prior_query.diagnostics.get('elapsed'),
                   **{k: amount(diag.get(k, 0)) for k in (
                       'new_regions', 'updated_regions', 'ambiguous_regions', 'new_entities',
                       'updated_entities', 'ambiguous_entity_proposals', 'proposal_budget_truncated',
                       'pruned_candidates', 'merged_region_ids', 'merged_entity_ids')}}
            del learned, diag
            release_observation()
            row['baseline_query_seconds'] = row['repeat_seconds'] = 0.
            current_phase = 'train:baseline_query'
            if step <= REPLAY_PROBES:
                initial_matches, baseline_diag = query_compact(batch_features, batch_mask)
                row['baseline_query_seconds'] = baseline_diag['query_seconds']
                probe_baselines[str(path)] = sorted({m['template_id'] for m in initial_matches})
            if step == 1:
                # Same episode can improve fractional success, but each evidence entry stays <= 1.
                current_phase = 'train:same_episode_repeat'
                before_repeat = memory_counts()
                synchronize_device(); repeat_start = time.perf_counter()
                repeated = learner.learn_view(batch_features, source_id=str(path),
                                              episode_id=episode, valid_mask=batch_mask)
                synchronize_device()
                row['repeat_seconds'] = time.perf_counter() - repeat_start
                repeat_stages = dict(repeated.diagnostics.get('stage_seconds', {}))
                if repeated.diagnostics.get('learning_status') == 'UNRESOLVED':
                    append_log({'phase': 'unresolved_repeat', 'step': step, 'path': str(path),
                                **repeated.diagnostics})
                    raise RuntimeError('Repeat search unfinished; repeat evidence was not committed.')
                del repeated
                release_observation()
                repeat_row = {'phase': 'same_episode_repeat', 'step': step, 'before': before_repeat,
                              'after': memory_counts(), 'repeat_seconds': row['repeat_seconds'],
                              'stage_seconds': repeat_stages, **graph_footprint_and_evidence(), **resources()}
                append_log(repeat_row); display(repeat_row)
            del batch_features, batch_view, batch_mask
            release_observation()
            current_phase = 'train:metrics'
            row.update(memory_counts()); row.update(resources())
            row['run_sampled_peak_rss_mib'] = monitor.peak
            if step == 1 or step % GRAPH_SIZE_EVERY == 0 or step == len(train_paths):
                row.update(graph_footprint_and_evidence())
            row['step_seconds'] = time.perf_counter() - start
            row['other_seconds'] = max(0., row['step_seconds'] - sum(row[k] for k in
                ('learn_seconds', 'encode_seconds', 'baseline_query_seconds', 'repeat_seconds')))
            learning_rows.append(row); append_log(row)
            if step == 1 or step % REPORT_EVERY == 0 or step == len(train_paths):
                display(row)

        # Frozen memory: first-image retention and disjoint held-out activation.
        before_query = memory_counts()
        entity_evidence_before = {eid: (e.version, e.evidence.raw_hits, e.evidence.support, e.evidence.opportunities)
                                  for eid, e in learner.gmem_iii.entity_nodes.items()}
        for phase, selection in [('replay', train_paths[:REPLAY_PROBES]), ('heldout', holdout_paths)]:
            for index, path in enumerate(selection, 1):
                current_path, current_phase = str(path), phase + ':query'
                batch_features, batch_view, batch_mask = load_dataset_image(path)
                matches, qdiag = query_compact(batch_features, batch_mask)
                ids = {m['template_id'] for m in matches}
                previous = set(probe_baselines.get(str(path), []))
                # Compare transformed-image roots to frozen untransformed retrieval, not object truth.
                dx, dy = DATASET_SHIFT
                shifted_features, _, shifted_mask = encode_image(
                    shifted(batch_view, dx, dy), shifted(batch_mask[None, None].float(), dx, dy))
                shifted_matches, shifted_diag = query_compact(shifted_features, shifted_mask)
                distances, eligible = [], 0
                for match in matches:
                    target = np.asarray(match['point']) + (dx, dy)
                    if not (0 <= target[0] < cfg.W and 0 <= target[1] < cfg.H):
                        continue
                    eligible += 1
                    candidates = [np.linalg.norm(np.asarray(m['point']) - target)
                                  for m in shifted_matches if m['template_id'] == match['template_id']]
                    if candidates:
                        distances.append(float(min(candidates)))
                row = {'phase': phase, 'path': str(path), **qdiag,
                       'initial_entity_ids': sorted(previous), 'final_entity_ids': sorted(ids),
                       'id_retention_proxy': len(ids & previous) / len(previous) if previous else None,
                       'translation_eligible_roots': eligible, 'translation_matched_roots': len(distances),
                       'translation_within_2px': sum(d <= 2 for d in distances),
                       'translation_error_median_px': float(np.median(distances)) if distances else None,
                       'shifted_query': shifted_diag, 'top_entities': sorted(matches, key=lambda m: -m['score'])[:8],
                       **resources()}
                evaluation_rows.append(row); append_log(row)
                if index <= 2:
                    fig, ax = plt.subplots(figsize=(6, 5))
                    ax.imshow(batch_view[0].permute(1, 2, 0).cpu().clamp(0, 1))
                    for match in row['top_entities']:
                        ax.scatter(*match['point'], c='red', marker='+', s=70)
                        ax.annotate(f"III:{match['template_id']} / {match['score']:.2f}", match['point'],
                                    color='yellow', fontsize=8)
                    ax.set_title(f'{phase}: {path.name}\nRed +: predicted entity root (not object labels)')
                    plt.show(); plt.close(fig)
                del batch_features, batch_view, batch_mask, shifted_features, shifted_mask
                if index % REPORT_EVERY == 0 or index == len(selection):
                    display({k: v for k, v in row.items() if k != 'top_entities'})
        entity_evidence_after = {eid: (e.version, e.evidence.raw_hits, e.evidence.support, e.evidence.opportunities)
                                 for eid, e in learner.gmem_iii.entity_nodes.items()}
        frozen_check = before_query == memory_counts() and entity_evidence_before == entity_evidence_after
        append_log({'phase': 'frozen_check', 'counts_and_entity_evidence_unchanged': frozen_check})
        run_status = 'completed' if frozen_check else 'query_mutation_detected'
    except (Exception, KeyboardInterrupt) as exc:
        run_status = 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed'
        append_log({'phase': 'error', 'status': run_status, 'at_phase': current_phase,
                    'path': current_path, 'error_type': type(exc).__name__, 'message': str(exc), **resources()})
        print(f'{run_status}: {current_phase}, {current_path}: {type(exc).__name__}: {exc}')
        # Stop after a failure; never count an incomplete run as a successful full-data test.

# Release partial observation/query tensors as well when an image failed.
for temporary_name in ('learned', 'repeated', 'diag', 'batch_features', 'batch_view', 'batch_mask',
                       'shifted_features', 'shifted_mask', 'entity_evidence_before', 'entity_evidence_after'):
    globals().pop(temporary_name, None)
release_observation()

summary = {'status': run_status, 'completed_train': len(learning_rows), 'planned_train': len(train_paths),
           'completed_evaluations': len(evaluation_rows), 'baseline': baseline_resources,
           'run_elapsed_seconds': time.perf_counter() - run_started, 'cuda_environment': CUDA_ENV,
           'final': resources(), 'run_sampled_peak_rss_mib': monitor.peak,
           'peak_rss_increment_from_baseline_mib': max(0, monitor.peak - baseline_resources['rss_mib']),
           'minimum_host_available_mib': monitor.minimum_available, 'sampler_error': monitor.error,
           **memory_counts()}
# Traversing a large graph itself has overhead; avoid it after OOM/interruption.
if run_status == 'completed':
    summary.update(graph_footprint_and_evidence())
for phase in ('replay', 'heldout'):
    rows = [r for r in evaluation_rows if r['phase'] == phase]
    total_roots = sum(r['translation_eligible_roots'] for r in rows)
    summary[phase] = {'images': len(rows),
                      'entity_activation_fraction': sum(r['entity_matches'] > 0 for r in rows) / len(rows) if rows else None,
                      'translation_within_2px_fraction': sum(r['translation_within_2px'] for r in rows) / total_roots if total_roots else None}
(run_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
display(summary)
entity_rows = [{'entity_id': eid, 'status': e.status, 'members': len(e.component_edges),
                'root_slot': e.root_slot, **e.evidence.summary()}
               for eid, e in sorted(learner.gmem_iii.entity_nodes.items())]
display(entity_rows[:40])
if learning_rows:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    steps = [r['step'] for r in learning_rows]
    for key in ('gmem_i', 'gmem_ii', 'gmem_iii', 'stable_entities'):
        axes[0, 0].plot(steps, [r[key] for r in learning_rows], label=key)
    axes[0, 0].set(title='Persistent graph growth', ylabel='Nodes')
    for key in ('entity_raw_hits', 'entity_support'):
        axes[0, 1].plot(steps, [r[key] for r in learning_rows], label=key)
    axes[0, 1].set(title='Entity evidence (merges/pruning can reduce totals)', ylabel='Evidence')
    for key in ('rss_mib', 'run_sampled_peak_rss_mib', 'cuda_allocated_mib', 'cuda_reserved_mib'):
        if key in learning_rows[0]:
            axes[1, 0].plot(steps, [r[key] for r in learning_rows], label=key)
    measured = [r for r in learning_rows if 'graph_cpu_estimate_mib' in r]
    axes[1, 0].plot([r['step'] for r in measured], [r['graph_cpu_estimate_mib'] for r in measured],
                    'o--', label='Graph CPU estimate')
    axes[1, 0].set(title='Memory: process / persistent graph / GPU', ylabel='MiB')
    for key in ('learn_seconds', 'step_seconds'):
        axes[1, 1].plot(steps, [r[key] for r in learning_rows], label=key)
    axes[1, 1].set(title='Per-image time (step includes probes / diagnostics)', ylabel='Seconds')
    for ax in axes.flat:
        ax.set_xlabel('Training image'); ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(run_dir / 'learning_resources.png', dpi=140)
    plt.show(); plt.close(fig)
if learning_rows and any(r.get('stage_seconds') for r in learning_rows):
    fig, ax = plt.subplots(figsize=(12, 5))
    stage_names = list(dict.fromkeys(k for r in learning_rows for k in r.get('stage_seconds', {})))
    bottom = np.zeros(len(learning_rows))
    for name in stage_names:
        values = np.asarray([r.get('stage_seconds', {}).get(name, 0.) for r in learning_rows])
        ax.bar([r['step'] for r in learning_rows], values, bottom=bottom, label=name)
        bottom += values
    ax.set(xlabel='Training image', ylabel='Seconds', title='learn_view stages (CUDA synchronized)')
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout(); fig.savefig(run_dir / 'learning_stages.png', dpi=140)
    plt.show(); plt.close(fig)
print('Run status:', run_status, '| Logs:', run_dir)


## 9. GmemIII 成员、共享从属与虚拟挂载

左图画实体坐标系中的成员锚点：星号为根角色，圆点为其他角色，标签同时标注角色与 GmemII id。
同一个 GmemII id 可在一个实体中出现多次，也可被多个实体引用。线条表示持久相对关系。
右图为按成员分组着色的虚拟叶点，不修改原 GmemII。编号仍代表角色，重合不等于可以合并实例。


In [ ]:
from nns.memorygraphs.graph_memorypool_onceoptimizer import entity_view
ENTITY_INDEX = 0
entity_ids = sorted(learner.gmem_iii.entity_nodes)
if not entity_ids:
    print('No entity candidates: inspect region matching, contacts, proposal budget and ambiguity.')
else:
    entity = learner.gmem_iii.entity_nodes[entity_ids[min(max(0, ENTITY_INDEX), len(entity_ids) - 1)]]
    view = entity_view(entity, learner.gmem_ii, learner.cfg)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for role, member in entity.component_edges.items():
        color = plt.get_cmap('tab10')(role % 10)
        marker = '*' if role == entity.root_slot else 'o'
        axes[0].scatter(member['dx'], member['dy'], c=[color], marker=marker, s=130)
        axes[0].annotate(f"slot {role} / II {member['sem_id']}", (member['dx'], member['dy']),
                         xytext=(4, 5), textcoords='offset points', fontsize=8)
        children = [s for s in view.slots if s['group'] == role]
        xy = np.asarray([s['xy'] for s in children])
        if len(xy):
            axes[1].scatter(xy[:, 0], xy[:, 1], c=[color], s=12, label=f'slot {role}')
    for (a, b), relation in entity.relation_edges.items():
        ma, mb = entity.component_edges[a], entity.component_edges[b]
        axes[0].plot([ma['dx'], mb['dx']], [ma['dy'], mb['dy']], 'k--', alpha=.4)
    for ax in axes:
        ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xlabel('relative x / pixels'); ax.set_ylabel('relative y / pixels')
    axes[0].set_title(f'GmemIII {entity.node_id}: members ({entity.status})')
    axes[1].set_title('Virtual leaf positions, grouped by member'); axes[1].legend()
    plt.tight_layout(); plt.show()
    display([{'role': role, 'semantic_id': m['sem_id'], 'offset': (m['dx'], m['dy']),
              'parent_entities': sorted(learner.gmem_ii.parent_entities[m['sem_id']]),
              **m['evidence'].summary()} for role, m in entity.component_edges.items()])


## 10. 冻结长期记忆：GposII／III、位图与候选预算

默认稀疏模式；`EXACT_HIERARCHY_QUERY=True` 对小图执行所有平移位置的密集对照，可能很慢。
`search_complete=False` 表示尚不能保证无漏检。位图用于候选传播，命中位数不是空间检索成功率。
结果分别报告覆盖率、残差和图层；红色 + 为实际检索峰。此处仍查询前面演示图，
它不一定属于第 8 节训练集，因此不叠加其他图片的写入位置作为真值。
查询不使用观察图真值作为匹配输入，也不增加任何长期支持计数。


In [ ]:
EXACT_HIERARCHY_QUERY = False
hierarchy_queries = []
cases = query_cases if 'query_cases' in globals() else {'original': (image_view, valid_mask[None, None], (0, 0))}
for name, (pixels, mask, translation) in cases.items():
    features, query_view, query_valid = encode_image(pixels, mask)
    before = {eid: (e.evidence.raw_hits, e.evidence.support, e.version)
              for eid, e in learner.gmem_iii.entity_nodes.items()}
    retrieved = learner.query_hierarchy(features, valid_mask=query_valid, exact=EXACT_HIERARCHY_QUERY)
    after = {eid: (e.evidence.raw_hits, e.evidence.support, e.version)
             for eid, e in learner.gmem_iii.entity_nodes.items()}
    hierarchy_queries.append((name, retrieved))
    diagnostics = {k: v for k, v in retrieved.diagnostics.items()
                   if k not in {'static_region_bits', 'static_entity_bits'}}
    diagnostics['query_hit_bits_hex'] = hex(diagnostics.pop('query_hit_bits', 0))
    diagnostics['sampled_evaluated_bits_hex'] = hex(diagnostics.pop('sampled_evaluated_bits', 0))
    print(name, 'persistent evidence unchanged:', before == after)
    display(diagnostics)
    display({'region_requirement_bits': [(sid, hex(bits)) for sid, bits in
              list(retrieved.diagnostics.get('static_region_bits', {}).items())[:8]],
             'entity_requirement_bits': [(eid, hex(bits)) for eid, bits in
              list(retrieved.diagnostics.get('static_entity_bits', {}).items())[:8]]})
    display([m.summary() for m in retrieved.entities[:20]])
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, matches, label in zip(axes, [retrieved.regions, retrieved.entities], ['GposII', 'GposIII']):
        ax.imshow(query_view[0].permute(1, 2, 0).cpu().clamp(0, 1))
        for match in matches[:30]:
            ax.scatter(*match.point, marker='+', c='red', s=60)
            ax.annotate(f'{match.template_id}: {match.score:.2f}', match.point,
                        xytext=(3, 3), textcoords='offset points', color='yellow', fontsize=7)
        ax.set_title(f'{name}: {label} verified peaks (first 30)')
        ax.set_xlabel('x / pixels'); ax.set_ylabel('y / pixels')
    fig.text(.5, .01, 'Red +: predicted root; labels: id / score; no object ground truth is supplied.', ha='center', fontsize=9)
    plt.tight_layout(rect=(0, .05, 1, 1)); plt.show()


## 11. 可选参数对照

默认关闭。启用后每组重新构建全部记忆，不使用前一组的图或激活状态。
先固定分割，仅比较采样预算与实例/原型策略；不要把增加节点数量直接当作效果更好。

In [ ]:
RUN_COMPARISONS = False
COMPARISONS = {
    'instance_baseline': {},
    'denser_samples': {'once_samples_per_region': 128, 'once_surface_cover_radius': 8.},
    'prototype_reuse': {'once_node_reuse_mode': 'prototype'},
}
comparison_rows = []
if RUN_COMPARISONS:
    for name, overrides in COMPARISONS.items():
        trial_cfg = make_config(overrides)
        trial = MultilevelCoordinator(trial_cfg)
        trial_report = trial.handle_new_view(inputs, valid_mask=valid_mask, learn=False)
        rows = check_written_geometry(trial_report, trial.gmem_ii)
        comparison_rows.append({'name': name, 'overrides': overrides,
                                **trial_report.summary(),
                                'max_geometry_error': max((r['position_error'] for r in rows), default=0.),
                                'collapsed_edges': sum(r['collapsed_instances'] for r in rows)})
    display(comparison_rows)
else:
    print('Parameter comparisons disabled.')

## 12. 保存结果

手动设置 `SAVE_RESULTS=True` 后保存摘要、配置、区域与节点映射、边坐标、检索结果和标签矩阵。
过程掩码只用于诊断，后续查询不得读取这些真值。Notebook 不自动覆盖已有结果目录，
每次保存生成独立时间戳目录。本次仅编译检查，所有输出均等待重新执行。

In [ ]:
SAVE_RESULTS = False


def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


if SAVE_RESULTS:
    from datetime import datetime
    destination = OUTPUT / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    destination.mkdir(parents=True, exist_ok=False)
    region_rows = [{'region_id': r.region_id, 'modality': r.modality_id, 'support_dim': r.support_dim,
                    'anchor': r.anchor, 'samples': r.samples, 'semantic_id': r.semantic_id,
                    'coverage_error': r.coverage_error, 'curve_error': r.curve_error,
                    'completed': r.completed, 'budget_truncated': r.budget_truncated,
                    'view_truncated': r.view_truncated, 'closed': r.closed, 'reasons': r.reasons}
                   for r in report.regions]
    payload = {'summary': report.summary(), 'config': report.config,
               'input': str(IMAGE_PATH) if IMAGE_PATH is not None else 'synthetic',
               'view_size': VIEW_SIZE, 'original_hw': original_hw,
               'advanced_scales': ADVANCED_SCALES if ENABLE_ADVANCED else [],
               'regions': region_rows,
               'sample_nodes': [{'modality': m, 'pixel': p, 'node_id': nid}
                                for (m, p), nid in report.sample_nodes.items()],
               'edges': geometry, 'queries': query_results, 'comparisons': comparison_rows,
               'rejection_examples': report.rejection_examples}
    (destination / 'report.json').write_text(json.dumps(json_ready(payload), ensure_ascii=False,
                                                     indent=2, allow_nan=False))
    arrays = {'valid_mask': report.valid_mask}
    arrays.update({f'labels_{m}': value for m, value in report.labels.items()})
    arrays.update({f'support_{m}': value.support for m, value in report.supports.items()})
    np.savez_compressed(destination / 'maps.npz', **arrays)
    # Persist exact pools separately from diagnostic coordinate truth.
    torch.save({'gmem_i': coordinator.gmem_i, 'gmem_ii': coordinator.gmem_ii,
                'gmem_iii': coordinator.gmem_iii}, destination / 'memory.pt')
    if 'learner' in globals():
        torch.save(learner.state_dict(), destination / 'hierarchy_memory.pt')
    print('Saved:', destination)
else:
    print('Saving disabled.')